# Emparejamiento de archivos del docente con el satélite y piloto KAN

Este notebook documenta cómo se incorporaron los tres archivos de localización suministrados por el docente, cómo se construyeron los pares estación–píxel y cómo se puso a prueba la arquitectura funcional con cabeza KAN sobre validación.

**Resultado metodológico:** estos archivos no agregan ni reemplazan precipitación. Aportan latitud, longitud y elevación para unir las curvas terrestres con la grilla satelital.

In [ ]:
# Verificación reproducible: no reinstala paquetes presentes ni instala sin autorización.
from importlib.util import find_spec
import subprocess
import sys

INSTALAR_FALTANTES = False  # Cambiar a True únicamente si se autoriza instalar en este entorno.
PAQUETES_REQUERIDOS = {'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib', 'openpyxl': 'openpyxl', 'torch': 'torch'}
faltantes = {modulo: paquete for modulo, paquete in PAQUETES_REQUERIDOS.items()
             if find_spec(modulo) is None}

if faltantes and INSTALAR_FALTANTES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *faltantes.values()])
    faltantes = {modulo: paquete for modulo, paquete in PAQUETES_REQUERIDOS.items()
                 if find_spec(modulo) is None}

if faltantes:
    comando = f"{sys.executable} -m pip install {' '.join(faltantes.values())}"
    raise ModuleNotFoundError(
        f"Faltan dependencias: {faltantes}. No se instalaron automáticamente. "
        f"Autorice la instalación o ejecute: {comando}"
    )

versiones = {modulo: __import__(modulo).__version__
             for modulo in PAQUETES_REQUERIDOS
             if hasattr(__import__(modulo), '__version__')}
print('Dependencias verificadas:', versiones)


## 1. Variables de localización disponibles

El CSV y el archivo Stata contienen latitud y longitud. El Excel contiene elevación en metros. Ninguna de estas variables se estima: si una estación no aparece en alguno de los archivos, queda excluida de la versión que requiere todas las covariables.

En el piloto se comparan tres versiones de la KAN:

- KAN con latitud, longitud y elevación.
- KAN con latitud y longitud.
- KAN sin variables de ubicación.

Esto permite medir por separado si la ubicación y la elevación aportan información.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'Datos').exists():
    ROOT = ROOT.parent
RESULTADOS_KAN = ROOT / 'KAN' / 'resultados'
CATALOGO = RESULTADOS_KAN / 'catalogo_localizacion_docente_preparado.csv'
TRAZABILIDAD = RESULTADOS_KAN / 'pares_estacion_pixel_docente.csv'
IDENTIFICADORES = RESULTADOS_KAN / 'identificadores_pares_kan_docente.csv'
RESULTADOS = RESULTADOS_KAN / 'piloto_kan_docente'

for ruta in [CATALOGO, TRAZABILIDAD, IDENTIFICADORES, RESULTADOS]:
    assert ruta.exists(), f'Falta: {ruta}'
print('Entradas y resultados localizados correctamente.')


Entradas y resultados localizados correctamente.

## 2. Archivos suministrados por el docente

- `CVC_Wheather_Stations_Long_Lat.csv`: fuente operativa de código, latitud y longitud.
- `CVC_Wheather_Stations_Long_Lat.dta`: verificación independiente del mismo catálogo en formato Stata.
- `CatalogoEstacionesElevacion.xlsx`: código y elevación.

El CSV y el Stata contienen exactamente los mismos 238 códigos. Las diferencias máximas entre sus coordenadas son del orden del redondeo numérico. El Excel contiene 229 códigos únicos, sin elevaciones faltantes.

Datos de precipitación nuevos usados: **ninguno**.

La unión se realiza mediante igualdad exacta entre `STATION_ID` y `CODIGO ESTACION`. No se emplea coincidencia aproximada por nombre.

In [2]:
catalogo = pd.read_csv(CATALOGO, dtype={'station_id': 'string'})
columnas = ['station_id', 'station_name', 'latitude', 'longitude',
            'elevation_m', 'source_coordinates', 'source_elevation']
catalogo.loc[catalogo['station_id'].isin(
    ['2612800104', '2622330105', '2623500101']), columnas].round(6)


station_id,station_name,latitude,longitude,elevation_m,source_coordinates,source_elevation
2612800104,AUJI,3.622057,-76.107780,1555.0,CVC_Wheather_Stations_Long_Lat.csv,CatalogoEstacionesElevacion.xlsx
2622330105,LOS CRISTALES,3.427707,-76.568455,1312.0,CVC_Wheather_Stations_Long_Lat.csv,CatalogoEstacionesElevacion.xlsx
2623500101,EL CANEY,3.929590,-76.418328,1433.0,CVC_Wheather_Stations_Long_Lat.csv,CatalogoEstacionesElevacion.xlsx


## 3. Controles sin estimación

Las coordenadas geográficas se leen directamente del CSV suministrado. El archivo Stata se utiliza para comprobar códigos y valores. La elevación se une por código exacto desde el Excel.

No se interpola, no se completa y no se estima ninguna variable de localización faltante.

La implementación reproducible está en `preparar_emparejamiento_docente.py`.

In [3]:
# Esta llamada reconstruye el catálogo preparado y la tabla estación–píxel.
# No descarga precipitación ni modifica los archivos terrestres crudos.
script_emparejamiento = ROOT / 'KAN' / 'preparar_emparejamiento_docente.py'
%run $script_emparejamiento


Cruce reproducible: 110 estaciones con coordenadas, elevación y cobertura satelital; 2.167 pares utilizables.

## 4. Relación con el preprocesamiento terrestre

El catálogo se cruza **después** de aplicar las reglas terrestres. Por tanto, la reducción posterior no representa una nueva limpieza por calidad; representa disponibilidad espacial para construir entradas satelitales.

In [4]:
embudo = pd.DataFrame([
    ['Después del preprocesamiento terrestre', 263, 4769],
    ['Con latitud y longitud suministradas', 177, 3345],
    ['Con coordenadas dentro de la grilla', 154, 2922],
    ['Con coordenadas y elevación dentro de la grilla', 110, 2167],
], columns=['etapa', 'estaciones', 'pares_estacion_anio'])
embudo


etapa,estaciones,pares_estacion_anio
Después del preprocesamiento terrestre,263,4769
Con latitud y longitud suministradas,177,3345
Con coordenadas dentro de la grilla,154,2922
Con coordenadas y elevación dentro de la grilla,110,2167


## 5. Relación con la grilla satelital

La grilla completa contiene 39 filas × 37 columnas = 1.443 píxeles por año. Se conserva completa para el EDA satelital.

Para la red se extrae el píxel cuyo **centro es el más cercano** a cada estación. Las 110 estaciones con todas las covariables ocupan 93 píxeles distintos porque algunas comparten celda. Los demás píxeles no se eliminan: simplemente carecen de una curva terrestre objetivo emparejada.

In [5]:
pares = pd.read_csv(IDENTIFICADORES, dtype={'station_id': 'string'})
pd.DataFrame([
    ['Estaciones emparejadas', pares['station_id'].nunique()],
    ['Píxeles diferentes', pares[['pixel_row', 'pixel_col']].drop_duplicates().shape[0]],
    ['Pares entrenamiento', (pares['particion'] == 'ENTRENAMIENTO').sum()],
    ['Pares validación', (pares['particion'] == 'VALIDACION').sum()],
    ['Pares prueba reservados', (pares['particion'] == 'PRUEBA').sum()],
    ['Distancia mediana al centro del píxel (km)', pares['distance_to_pixel_center_km'].median()],
    ['Distancia máxima al centro del píxel (km)', pares['distance_to_pixel_center_km'].max()],
], columns=['control', 'valor'])


control,valor
Estaciones emparejadas,110.000
Píxeles diferentes,93.000
Pares entrenamiento,1436.000
Pares validación,464.000
Pares prueba reservados,267.000
Distancia mediana al centro del píxel (km),2.292
Distancia máxima al centro del píxel (km),3.872


### Distribución espacial

![Estaciones emparejadas y centros de píxel](KAN/resultados/piloto_kan_docente/mapa_emparejamiento.png)

## 6. Arquitectura puesta a prueba

```text
72 valores satelitales
        ↓ ajuste sobre 11 B-splines elegidas con entrenamiento
11 coeficientes satelitales
        ↓ 8 neuronas funcionales
8 características de la forma anual
        ↓ se concatenan latitud, longitud y elevación, según la versión
Cabeza lineal, MLP o KAN B-spline
        ↓
6 puntajes FPCA terrestres
        ↓ decodificador FPCA fijo
8 coeficientes terrestres
        ↓ evaluación sobre 365 días
Curva terrestre reconstruida
```

Los días faltantes terrestres no se rellenan antes de construir la curva. Los coeficientes B-spline se ajustan usando únicamente los días observados de cada estación–año.

In [6]:
resumen = json.loads((RESULTADOS / 'resumen_piloto.json').read_text(encoding='utf-8'))
pd.DataFrame([
    ['B-splines para la curva satelital', resumen['bases_satelite']],
    ['B-splines para la curva terrestre', resumen['bases_terrestre']],
    ['Neuronas funcionales', 8],
    ['Componentes FPCA', resumen['componentes_fpca']],
    ['Varianza FPCA acumulada', resumen['varianza_fpca_acumulada']],
    ['Ancho oculto KAN', resumen['kan_hidden']],
    ['Ancho oculto MLP', resumen['mlp_hidden']],
    ['Parámetros cabeza KAN', resumen['parametros_cabeza_kan']],
    ['Parámetros cabeza MLP', resumen['parametros_cabeza_mlp']],
], columns=['decisión', 'valor'])


decisión,valor
B-splines para la curva satelital,11.0000
B-splines para la curva terrestre,8.0000
Neuronas funcionales,8.0000
Componentes FPCA,6.0000
Varianza FPCA acumulada,0.9525
Ancho oculto KAN,9.0000
Ancho oculto MLP,43.0000
Parámetros cabeza KAN,780.0000
Parámetros cabeza MLP,780.0000


## 7. Protocolo sin fuga de información

- Entrenamiento: 1981–1995.
- Validación: 1996–2000.
- Prueba reservada: 2001–2003; no se consultó en este piloto.
- Media y desviación satelital: calculadas solo con entrenamiento.
- Estandarización de latitud, longitud y elevación: calculada solo con entrenamiento.
- FPCA terrestre: ajustada solo con entrenamiento.
- MLP y KAN con las tres covariables: exactamente 780 parámetros en la cabeza.
- Misma semilla, orden de lotes, máximo de épocas y parada temprana.

La comparación usa una sola semilla y solo sirve como prueba funcional. Una conclusión final requiere varias semillas.

In [7]:
# Reentrena los cuatro modelos y actualiza métricas y gráficas.
%run piloto_kan_docente.py


Piloto ejecutado correctamente. La partición de prueba permaneció reservada.

## 8. Resultados en validación

`RMSE observado` compara la predicción con los días terrestres disponibles. `RMSE objetivo funcional` compara dos curvas suavizadas en el mismo soporte, por lo que es menor y no debe confundirse con el error diario observado.

In [8]:
metricas = pd.read_csv(RESULTADOS / 'metricas_validacion.csv')
metricas[['modelo', 'MAE_observado_mm', 'RMSE_observado_mm',
          'RMSE_objetivo_funcional_mm', 'sesgo_total_anual_mm',
          'parametros_cabeza', 'epocas']].round(3)


modelo,MAE_observado_mm,RMSE_observado_mm,RMSE_objetivo_funcional_mm,sesgo_total_anual_mm,parametros_cabeza,epocas
KAN + lat/lon,5.029,10.618,1.452,-92.691,771,102
KAN + lat/lon/elevación,5.021,10.663,1.521,-125.107,780,56
MLP + lat/lon/elevación,5.011,10.740,1.617,-180.604,780,49
KAN sin ubicación,5.055,10.744,1.709,-162.934,753,30
Lineal + lat/lon/elevación,5.049,10.812,1.865,-199.913,72,44


### Curvas de validación

![Pérdida funcional en validación](KAN/resultados/piloto_kan_docente/curvas_validacion.png)

### Ejemplos no seleccionados por conveniencia

Se muestran curvas situadas aproximadamente en los percentiles 25, 50 y 75 del error del mejor modelo de validación.

![Ejemplos de reconstrucción](KAN/resultados/piloto_kan_docente/ejemplos_reconstruccion_validacion.png)

## 9. Diagnóstico del cuello de botella FPCA

El contrato provisional de la arquitectura restringe la salida a 4–6 componentes. Con las ocho bases terrestres previamente seleccionadas, seis componentes explican aproximadamente el 95,25 % de la variación funcional de entrenamiento.

In [9]:
fpca = pd.read_csv(RESULTADOS / 'varianza_fpca_train.csv')
fpca.head(8).round(4)


componente,varianza_explicada,varianza_acumulada
1,0.6408,0.6408
2,0.1141,0.7549
3,0.0757,0.8305
4,0.0529,0.8835
5,0.0372,0.9207
6,0.0318,0.9525
7,0.0243,0.9768
8,0.0232,1.0000


## 10. Conclusiones del piloto

1. El emparejamiento espacial es reproducible y no requiere estimar coordenadas ni elevación.
2. La KAN entrena con datos reales y reconstruye curvas terrestres.
3. Las diferencias entre MLP y KAN son pequeñas con una sola semilla; no se puede declarar un ganador.
4. La KAN con latitud y longitud obtuvo el menor RMSE observado en esta ejecución. Añadir elevación no mejoró ese indicador, por lo que su aporte debe evaluarse con varias semillas.
5. La versión sin ubicación y la cabeza lineal fueron inferiores en varios indicadores, lo que sugiere una contribución espacial y no lineal que todavía debe confirmarse.
6. Seis componentes FPCA superan el 95 % de varianza explicada de entrenamiento y son suficientes para este piloto.

## 11. Limitaciones que deben quedar explícitas

- Las unidades físicas y el posible techo de 200 del satélite siguen pendientes.
- Varias estaciones comparten píxel.
- La evaluación actual es temporal; una evaluación espacial debe agrupar por píxel para impedir que estaciones de una misma celda queden en particiones diferentes.
- La prueba 2001–2003 continúa cerrada.
- Una sola semilla demuestra funcionamiento, no superioridad estadística.